In [1]:
import numpy as np
import pickle as pkl
from hdbscan import HDBSCAN
from umap import UMAP

# Load artifacts

In [2]:
X_embed = np.load('../data/processed/embeddings.npy')

with open('../data/processed/kmeans_model.pkl', 'rb') as file:
    km = pkl.load(file)

print('Artifacts loaded successfully ✅')

Artifacts loaded successfully ✅


# Dimensionality reduction: UMAP

In [3]:
umap_reducer = UMAP(
    n_neighbors=15,
    n_components=2,
    min_dist=0.1,
    metric='cosine',
    random_state=42
)

X_umap = umap_reducer.fit_transform(X_embed)

/opt/miniconda3/envs/myenv/lib/python3.14/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/opt/miniconda3/envs/myenv/lib/python3.14/site-packages/umap/spectral.py:548: UserWarning: Spectral initialisation failed! The eigenvector solver
failed. This is likely due to too small an eigengap. Consider
adding some noise or jitter to your data.

Falling back to random initialisation!
  warn(
/opt/miniconda3/envs/myenv/lib/python3.14/site-packages/umap/spectral.py:548: UserWarning: Spectral initialisation failed! The eigenvector solver
failed. This is likely due to too small an eigengap. Consider
adding some noise or jitter to your data.

Falling back to random initialisation!
  warn(


# Fit: HDBSCAN

Because your SentenceTransformer vectors are L2-normalized, Euclidean distance mathematically mirrors Cosine similarity, making HDBSCAN highly effective out of the box. min_cluster_size=15 tells the algorithm: "Do not create a cluster unless you find at least 15 tickets tightly packed together."

In [4]:
clusterer = HDBSCAN(
    min_samples=5,
    min_cluster_size=15,
    metric='euclidean',
    cluster_selection_method='eom'
)

hdbscan_labels = clusterer.fit_predict(X_umap)
hdbscan_labels

array([237, 237,  -1, ..., 175, 258,  96], shape=(26872,))

In [5]:
unique_labels = len(np.unique(hdbscan_labels))
unique_labels - (1 if -1 in hdbscan_labels else 0) # subtracting value 1 (Not a list comprehension) if -1 labels (noise) are present.

576

# Noise ratio

In [6]:
n_noise = list(hdbscan_labels).count(-1)
# noise ratio
noise_ratio = n_noise / unique_labels
noise_ratio

3.835355285961872